In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch
device= "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Text Cleaning


In [3]:
import re
train= pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(train.isnull().sum().sum())  #no nulls values

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\-\./\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["cleaned_prompt"] = train["prompt"].apply(clean_text)
for opt in ["A","B","C","D","E"]:
    train[f"cleaned_{opt}"] = train[f"{opt}"].apply(clean_text)
train.head(5)

0


,id,prompt,A,B,C,D,E,answer,cleaned_prompt,cleaned_A,cleaned_B,cleaned_C,cleaned_D,cleaned_E
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


In [4]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

# Baseline Model


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

texts = []
labels = []
options = ["A","B","C","D","E"]
for i in range(len(train)):
    prompt = train["cleaned_prompt"][i]
    answer = train["answer"][i]
    for opt in options:
        text = prompt + " " + train[f"cleaned_{opt}"][i]
        texts.append(text)
        labels.append(1 if answer == opt else 0)

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
y_pred = lr.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

rows_per_q = 5
pred_opt = []
true_ans = []
for i in range(0, len(y_test), rows_per_q):
    probs = lr.predict_proba(X_test_tfidf[i:i+rows_per_q])[:,1]
    ranked = np.argsort(-probs)
    pred_opt.append([options[j] for j in ranked[:3]])
    correct_idx = np.argmax(y_test[i:i+rows_per_q])
    true_ans.append(options[correct_idx])

def map3(true_answers, predicted_options, k=3):
    score = 0
    for actual, preds in zip(true_ans, pred_opt):
        if actual in preds[:k]:
            score += 1.0 / (preds.index(actual) + 1)
    return score / len(true_ans)
map3_score = map3(true_ans, pred_opt, k=3)
print("MAP@3:", map3(true_ans, pred_opt, k=3))

Accuracy: 0.8045
F1 Score: 0.09280742459396751
MAP@3: 0.625416666666667


In [ ]:
!pip install wandb

In [17]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wandb")
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f3004475 (24f3004475-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [22]:
wandb.init(
    project="dlgenai-t226",
    name="baseline-model",
    config={"Model": "Logistic Regression",
            "Random Seed": 42,
            "Train/Test Split": "80/20" })
wandb.log({"accuracy": accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "map@3": map3_score })
wandb.finish()

accuracy,▁
f1_score,▁
map@3,▁
accuracy,0.8045
f1_score,0.09281
map@3,0.62542


# RoBERTa tokenization & training

In [6]:
import random
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.cuda.manual_seed_all(42)

In [7]:
from transformers import AutoTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

train_df, val_df = train_test_split(train, test_size=0.2,random_state=42, stratify=train["answer"])

def create_pairs(df):
    rows = []
    for _, row in df.iterrows():
        prompt = row["cleaned_prompt"]
        for opt in options:
            rows.append({"question_id": row["id"],
                    "text": prompt + " </s> " + row[f"cleaned_{opt}"],
                    "label": 1 if row["answer"] == opt else 0, "option": opt})
    return pd.DataFrame(rows)
train_pairs = create_pairs(train_df)
val_pairs = create_pairs(val_df)

train_dataset = Dataset.from_pandas(train_pairs)
val_dataset = Dataset.from_pandas(val_pairs)

In [8]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True,padding="max_length",max_length=256 )
    
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])
val_dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
training_args = TrainingArguments(output_dir="./roberta_results",
                            eval_strategy="epoch",
                            save_strategy="epoch",
                            learning_rate=2e-5,
                            per_device_train_batch_size=16,
                            per_device_eval_batch_size=16,
                            num_train_epochs=3,
                            weight_decay=0.01,
                            logging_steps=100,
                            load_best_model_at_end=True, seed=42)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=750, training_loss=0.821944943745931, metrics={'train_runtime': 627.5182, 'train_samples_per_second': 38.246, 'train_steps_per_second': 1.195, 'total_flos': 3157332664320000.0, 'train_loss': 0.821944943745931, 'epoch': 3.0})

In [25]:
preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy:", acc)
print("F1 Score:", f1)

probs = torch.softmax(torch.tensor(preds.predictions), dim=1).numpy()
positive_probs = probs[:, 1]
rows_per_q = 5
pred_opt = []
true_ans = []

def map3(actual, predicted):
    score = 0
    for a, p in zip(actual, predicted):
        if a in p:
            score += 1 / (p.index(a) + 1)
    return score / len(actual)
    
for i in range(0, len(positive_probs), rows_per_q):
    q_probs = positive_probs[i:i+5]
    ranked_indices = np.argsort(q_probs)[::-1]
    top3 = [options[j] for j in ranked_indices[:3]]
    pred_opt.append(top3)
    correct_index = np.argmax(y_true[i:i+5])
    true_ans.append(options[correct_index])

map3_r = map3(true_ans, pred_opt)
print("MAP@3:", map3_r)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.916
F1 Score: 0.7633802816901408
MAP@3: 0.9429166666666666


In [26]:
wandb.init(project="dlgenai-t226",
            name="roberta-pretrained",
           config={"Model": "RoBERTa",
                    "Epochs": 3,
                    "Batch Size": 16,
                    "Learning Rate": 2e-5} )
wandb.log({ "accuracy": acc,
            "f1_score": f1,
             "map@3": map3_r})
wandb.finish()

accuracy,▁
f1_score,▁
map@3,▁
accuracy,0.916
f1_score,0.76338
map@3,0.94292


In [10]:
submission = pd.DataFrame({
    "ID": range(1, len(pred_opt)+1),
    "Prediction": pred_opt
})

submission.to_csv("submission.csv", index=False)

In [11]:
# import nltk
# from gensim.models import Word2Vec
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# nltk.download("punkt")
# train["prompt_token"] = train["cleaned_prompt"].apply(nltk.word_tokenize)
# for opt in ["A","B","C","D","E"]:
#     train[f"{opt}_token"] = train[f"cleaned_{opt}"].apply(nltk.word_tokenize)

# text = train["cleaned_prompt"].tolist()
# for opt in ["A","B","C","D","E"]:
#     text += train[f"cleaned_{opt}"].tolist()

# tfidf = TfidfVectorizer()
# tfidf.fit(text)

# i = 0
# p_vec = tfidf.transform([train.loc[i,"cleaned_prompt"]])
# o_vec = [tfidf.transform([train.loc[i,f"cleaned_{opt}"]]) for opt in ["A","B","C","D","E"]]

# cos_sim = [cosine_similarity(p_vec, opt)[0][0] for opt in o_vec]
# print(cos_sim)

# all_tokens = train["prompt_token"].tolist()
# for opt in ["A","B","C","D","E"]:
#     all_tokens += train[f"{opt}_token"].tolist()

# w2v = Word2Vec(sentences=all_tokens, vector_size=100, window=5, min_count=1, workers=4)
# def avg_vector(tokens, model):
#     vecs = [model.wv[w] for w in tokens if w in model.wv]
#     if len(vecs) > 0:
#         return np.mean(vecs, axis=0) 
#     else:
#         return np.zeros(model.vector_size)
        
# i = 0
# p_avg = avg_vector(train.loc[i,"prompt_token"], w2v)
# o_avg = [avg_vector(train.loc[i,f"{opt}_token"], w2v) for opt in ["A","B","C","D","E"]]

# w2v_sim = [cosine_similarity([p_avg],[o])[0][0] for o in o_avg]
# print(w2v_sim)

# def predict_top3(prompt_tokens, option_tokens, model):
#     p_vec = avg_vector(prompt_tokens, model)
#     sim = [cosine_similarity([p_vec],[avg_vector(opt, model)])[0][0] for opt in option_tokens]
#     ranked = np.argsort(sim)[::-1]
#     return [chr(65 + idx) for idx in ranked[:3]]
    
# def mapk(actual, predicted, k=3):
#     score = 0.0
#     n = len(actual)

#     for i in range(n):
#         a = actual[i]
#         p = predicted[i]
#         if a in p:
#             idx = p.index(a)
#             score += 1.0 / (idx + 1)
#     return score / n
    
# actual_ans = train["answer"].tolist()[:500]
# pred_ans = []

# for i in range(500):
#     prompt_tokens = train.loc[i,"prompt_token"]
#     option_tokens_list = [train.loc[i,f"{opt}_token"] for opt in ["A","B","C","D","E"]]
#     preds = predict_top3(prompt_tokens, option_tokens_list, w2v)
#     pred_ans.append(preds)

# print(mapk(actual_ans, pred_ans))

# RAG Inference

In [12]:
# from sentence_transformers import SentenceTransformer
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# embedder = SentenceTransformer("all-MiniLM-L6-v2")
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
# classifier = pipeline("zero-shot-classification", model=model, tokenizer=tokenizer)

# def retrieve_context(question, opts, k=3):
#     q_emb = embedder.encode([question], convert_to_numpy=True).astype("float32")
#     opt_emb = embedder.encode(opts, convert_to_numpy=True).astype("float32")
#     scores = np.dot(opt_emb, q_emb.T).flatten()
#     ranked = np.argsort(-scores)
#     return [opts[i] for i in ranked[:k]]
    
# pred_opt = []
# true_ans = []
# opt = ["A","B","C","D","E"]
# rows_per_question = 5
# for i in range(0, len(dataset["test"]), rows_per_question):
#     q_text = dataset["test"][i]["text"]
#     opts = [dataset["test"][i+j]["text"] for j in range(rows_per_question)]
#     retrieved = retrieve_context(q_text, opts, k=3)
#     result = classifier(q_text, candidate_labels=retrieved)
#     ranked_options = []
#     for label in result["labels"]:
#         if label in opts:
#             ranked_options.append(options[opts.index(label)])
#     pred_opt.append(ranked_options)
#     correct_idx = np.argmax([dataset["test"][i+j]["label"] for j in range(rows_per_question)])
#     true_ans.append(options[correct_idx])
# mapk(true_ans, pred_opt, k=3)

# FAISS Retrieval

In [13]:
# !pip install faiss-cpu -q
# import faiss

# texts = [row["text"] for row in dataset["train"]]
# embds = embedder.encode(texts)
# dim = embds.shape[1]
# index = faiss.IndexFlatL2(dim)
# index.add(embds)
# def retrieve_context(question,opts, k=3):
#     q_emb = embedder.encode([question], convert_to_numpy=True).astype("float32")
#     opt_emb = embedder.encode(opts, convert_to_numpy=True).astype("float32")
#     dim = opt_emb.shape[1]
#     index = faiss.IndexFlatL2(dim)
#     index.add(opt_emb)
#     D, I = index.search(q_emb, k)
#     return [opts[idx] for idx in I[0]]

# for i in range(0, len(dataset["test"]), rows_per_question):
#     q_text = dataset["test"][i]["text"]
#     opts = [dataset["test"][i+j]["text"] for j in range(rows_per_question)]
#     retrieved = retrieve_context(q_text,opts, k=3)
#     result = classifier(q_text, candidate_labels=retrieved)
#     ranked_options = []
#     for lbl in result["labels"]:
#         if lbl in opts:
#             ranked_options.append(options[opts.index(lbl)])
#     pred_opt.append(ranked_options)
#     correct_idx = np.argmax([dataset["test"][i+j]["label"] for j in range(rows_per_question)])
#     true_ans.append(options[correct_idx])
# mapk(true_ans, pred_opt, k=3)

In [14]:
# sample= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
# sample.to_csv('submission.csv', index=False)